# Notebook 2 – Map files: from pixels to meters

`map_saver_cli` saves a map as two files: an **image** (`.pgm`) and a
**description** (`.yaml`). In this notebook you open both yourself and answer
the questions Nav2 asks:

1. What is in the `.yaml` and the `.pgm`?
2. Which cells are free, occupied, unknown?
3. How do you convert between pixels and meters?
4. **Can the robot reach a goal?** How do robot size and inflation from
   `robot.yaml` decide that?

Tutorial: [chapter 8 – what is stored where?](../../TUTORIAL_SLAM_NAVIGATION.md#8-what-is-stored-where),
[section 6.5 – how big is the robot?](../../TUTORIAL_SLAM_NAVIGATION.md#65-how-big-is-the-robot)

**How to use this notebook**

- Run the cells from top to bottom with **Shift + Enter**.
- `LIVE = False` uses the sample data in `sample_data/` – no robot needed.
- `LIVE = True` takes the data from the running robot. Start Jupyter from a
  terminal in which ROS 2 and the workspace are sourced (see `README.md` in this folder).
- Change values, run the cells again, look at what changes. You can't break anything.

Here, `LIVE = True` simply means: use **your own** saved map `~/maps/my_map.yaml`
instead of the sample map. No robot needs to be running.

In [ ]:
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
from nb_helpers import *

LIVE = False     # True: use your own map ~/maps/my_map.yaml
MAP_YAML = MY_MAP if LIVE else SAMPLE_MAP
print('map file:', MAP_YAML)

## 1. The `.yaml` file

In [ ]:
print(open(MAP_YAML).read())

| Key | Meaning |
| --- | --- |
| `image` | the image file that belongs to this map |
| `resolution` | meters per pixel (0.05 = 5 cm) |
| `origin` | map coordinates (x, y, yaw) of the **lower left** corner of the image |
| `occupied_thresh`, `free_thresh` | how dark a pixel must be to count as occupied / how light to count as free |

## 2. The `.pgm` image

A `.pgm` is the simplest image format there is: a short text header, then one
byte (0–255) per pixel. `map_saver_cli` writes only three gray values.

In [ ]:
grid = GridMap(MAP_YAML)
print(f'image size: {grid.width} x {grid.height} pixels')
print(f'that is   : {grid.width * grid.resolution:.2f} m x {grid.height * grid.resolution:.2f} m')

values, counts = np.unique(grid.image, return_counts=True)
for v, c in zip(values, counts):
    print(f'pixel value {v:3d}: {c:6d} pixels')

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(grid.image, cmap='gray', vmin=0, vmax=255)
ax.set_title('raw image: row 0 is at the TOP, axes in pixels')
ax.set_xlabel('column'); ax.set_ylabel('row')
plt.show()

## 3. Free, occupied, unknown

`map_server` turns every pixel into an occupancy probability
$p = (255 - \text{value}) / 255$ (white → 0, black → 1) and then:

- $p >$ `occupied_thresh` → **occupied** (100 in `/map`)
- $p <$ `free_thresh` → **free** (0 in `/map`)
- otherwise → **unknown** (−1 in `/map`)

In [ ]:
cell_area = grid.resolution ** 2
for name, value in [('free', FREE), ('occupied', OCCUPIED), ('unknown', UNKNOWN)]:
    n = int(np.sum(grid.cells == value))
    print(f'{name:9s}: {n:6d} cells = {n * cell_area:6.2f} m^2')

### A trap: `free_thresh`

`map_saver_cli` writes unknown cells as gray value **205**. That is
$p = (255 - 205) / 255 = 0.196$. With Nav2's default `free_thresh: 0.25`,
$0.196 < 0.25$ – so when the map is loaded again, **unknown becomes free**!
That's why the tutorial saves maps with `--free 0.196`. Let's see the difference:

In [ ]:
p = (255.0 - grid.image) / 255.0
for free_thresh in [0.25, 0.196]:
    unknown = (p >= free_thresh) & (p <= grid.info['occupied_thresh'])
    print(f'free_thresh = {free_thresh}: {int(unknown.sum()):5d} unknown cells')

## 4. The map in meters

Now we draw the classified map with axes in **meters**. The green triangle is
the start pose (0, 0) of the mapping run, the blue dot is `origin` – the lower
left corner of the image.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
show_map(ax, grid, 'map in meters')
ax.plot(grid.origin_x, grid.origin_y, 'bo', markersize=8, label='origin (lower left corner)')
ax.grid(True, alpha=0.3); ax.legend(loc='upper right')
plt.show()

## 5. Pixels ↔ meters

$$\text{column} = \left\lfloor \frac{x - \text{origin}_x}{\text{resolution}} \right\rfloor
\qquad
\text{row} = (\text{height} - 1) - \left\lfloor \frac{y - \text{origin}_y}{\text{resolution}} \right\rfloor$$

The row is "flipped", because in the image row 0 is at the **top**, but in the
map y grows **upwards**.

In [ ]:
row, col = grid.world_to_pixel(0.0, 0.0)
print(f'start pose (0, 0) is pixel row {row}, column {col}')
print('back to meters (cell center):', grid.pixel_to_world(row, col))

x, y = 2.0, 1.0
print(f'\n({x}, {y}) m -> pixel {grid.world_to_pixel(x, y)} -> cell type',
      ['FREE', 'UNKNOWN', 'OCCUPIED'][grid.cell_at(x, y)])

**Question:** the cell (row, column) = (0, 0) is the top left pixel. Which map
coordinates does it have? Compute it by hand first, then check with
`grid.pixel_to_world(0, 0)`.

## 6. Can the robot reach a goal?

Nav2 does not only ask "is the cell free?". The robot is not a point – it has
a size. Nav2 asks: **how far is the robot center from the nearest obstacle?**

- closer than robot radius + safety margin → the robot would touch it → **lethal**
- closer than the inflation distance → allowed but expensive → robot keeps away, drives slowly
- farther → fine

The values come from `config/robot.yaml`. First the distance from every cell
to the nearest obstacle:

In [ ]:
config = load_robot_config()
outer = robot_outer_radius(config)
inflation = config['navigation']['inflation_cm'] / 100
print(f'robot size incl. safety margin: {outer * 100:.1f} cm')
print(f'inflation distance            : {inflation * 100:.1f} cm')

distances = grid.distance_to_obstacle(max_distance=1.0)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(np.where(grid.cells == FREE, distances, np.nan), extent=grid.extent(),
               origin='upper', cmap='viridis')
fig.colorbar(im, ax=ax, label='distance to the nearest obstacle [m] (capped at 1 m)')
ax.set_title('distance to obstacles (free cells only)'); ax.set_aspect('equal')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
plt.show()

Now let's check some goals with `goal_check()` (see `nb_helpers.py`). Add your own!

In [ ]:
goals = [(0.0, 0.0), (1.0, 0.5), (2.0, 0.8), (-0.9, 0.0), (1.0, 1.45), (3.0, 0.65), (4.0, 1.0)]
for gx, gy in goals:
    ok, text = goal_check(grid, distances, gx, gy, config)
    print('OK  ' if ok else 'NO  ', text)

## 7. Where can the robot center go?

We color every free cell by what Nav2 thinks about it. This shows directly
which gaps the robot can pass: in the sample map, the **door** in the wall at
x = 3 m is 50 cm wide, the gap at x = 1 m only 30 cm.

In [ ]:
zones = np.zeros(grid.cells.shape + (3,))
zones[grid.cells == UNKNOWN] = (0.7, 0.7, 0.7)
zones[grid.cells == OCCUPIED] = (0, 0, 0)
free = grid.cells == FREE
zones[free & (distances > inflation)] = (1, 1, 1)                          # fine
zones[free & (distances <= inflation) & (distances > outer)] = (1, 0.75, 0.3)   # expensive
zones[free & (distances <= outer)] = (0.9, 0.2, 0.2)                       # lethal

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(zones, extent=grid.extent(), origin='upper', interpolation='nearest')
ax.plot(0, 0, 'g^', markersize=10)
ax.set_title('robot center: white = fine, orange = expensive, red = lethal')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.set_aspect('equal')
plt.show()
print('minimum gap width the robot can pass:', f'{2 * outer * 100:.0f} cm')

Where the red zones from both sides of a gap touch, the robot can **not** pass.
That's also why goals near walls fail with `ABORTED`.

## Exercises

1. In `config/robot.yaml`, change `safety_margin_cm` to 6 and re-run sections
   6 and 7 (re-run the cell that loads `config`!). Can the robot still pass the door?
2. Add a footprint that sticks out at the front, e.g.
   `footprint_cm: [[25, 15], [25, -15], [-17, -15], [-17, 15]]`.
   What happens to the red zones? (Then set it back to `[]`.)
3. Set `inflation_cm` to 50. Which goals in section 6 change from "fine" to
   "inflation zone"?
4. Open your own map's `.yaml`. Which `free_thresh` does it have? If it is
   `0.25`, change it to `0.196` (with a text editor) – then unknown areas stay unknown.
5. Set `LIVE = True` and repeat everything with **your own** map. Where is the
   narrowest passage the robot can still drive through?